# Encoder router — shakedown (dataset v2)

Runs the d63 prototype end to end: build the corpus-branch artifacts, assemble the
training table, smoke one fold, then sweep every arm through leave-one-lane-out CV.

**Every number in this notebook is shakedown tier.** The catalog is stale in columns
several cell predicates read (d56e/d62 re-extraction owed), so cell targets are wrong
for digit-bearing queries. The arm comparison of record waits for the v3 dataset build.

Prerequisites: `poetry install` (torch + lightgbm are new) and a populated
`labels.parquet` — it carries query text and cell itself.

In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import numpy as np
import pandas as pd

from encoder_router import (
    CorpusProfile,
    GoldDocProfile,
    NgramSvd,
    QueryEmbeddings,
    TrainingTable,
)
from encoder_router.evaluate import ARMS, BGE, E5, Arm, LaneCV, run_arms
from encoder_router.targets import OUT_DIR

pd.set_option("display.width", 160)
table = TrainingTable()

In [17]:
table.route_targets()

dropping 13 labelled rows without text


,sparse_only,dense_only
0,1.0,1.0
1,1.0,0.0
2,1.0,1.0
3,1.0,0.0
4,1.0,0.0
...,...,...
52340,NaN,NaN
52341,1.0,0.0
52342,NaN,NaN
52343,NaN,NaN


## 1 · The training frame

Acceptability labels joined with query text and cell. Expect a printed count of
labelled rows dropped for missing text — rows the selection no longer carries.

In [19]:
frame = table.frame
print(f"{len(frame):,} rows across {frame['dataset'].nunique()} lanes")
display(frame["shape"].value_counts())
ok_columns = [c for c in frame.columns if c.startswith("ok_")]
display(frame[ok_columns].mean().rename("acceptability rate"))
frame.groupby("dataset").size().sort_values(ascending=False)

52,345 rows across 42 lanes


shape
routes_differ    25550
all_tied         16132
all_zero         10663
Name: count, dtype: int64

ok_dense_only     0.875942
ok_pure_rrf       0.874958
ok_sparse_only    0.783192
Name: acceptability rate, dtype: Float64

dataset
scirgen-geo-en                          9714
webfaq-eng                              8612
orcas                                   7547
gooaq                                   7075
msmarco-passage-dev                     3816
crumb-legal-qa                          2657
clerc                                   2556
rarb-math                               2064
lotte-technology-forum                  1288
quest                                   1192
crumb-code-retrieval                     986
miracl-en-dev                            515
rarb-code                                485
lotte-technology-search                  398
limit                                    352
dbpedia-entity                           279
crumb-set-operation-entity-retrieval     212
freshstack-langchain                     201
freshstack-laravel                       184
beir-nfcorpus                            165
antique                                  159
freshstack-angular                       129
br

### 1b · Label hygiene: the min_relevance audit

Graded-qrel lanes where `min_relevance = 1` count weakly-related documents as
full successes (the nfcorpus "aneurysm → L-citrulline" tie) — every route finds
*some* weak positive and the row ties. High `share_grade_1` + high `tie_share`
marks a lane whose ties are manufactured by the relevance floor, not by
retrieval. Raising a lane's `min_relevance` in `lanes.py` changes its labels,
so that lane must be relabelled afterwards:
`poetry run python src/scripts/label_routes.py --only <lane> --force`.

In [20]:
from pathlib import Path

from hybrid_search_rrf_dataset.lanes import LANES

records = []
for lane in sorted(frame["dataset"].unique()):
    source = LANES[lane].source.name if lane in LANES else lane
    qrels = pd.read_parquet(Path("data") / source / "qrels.parquet")
    grades = qrels["relevance"].value_counts().sort_index()
    if grades.index.max() <= 1:
        continue
    lane_rows = frame[frame["dataset"] == lane]
    records.append({
        "lane": lane,
        "min_relevance": LANES[lane].min_relevance if lane in LANES else 1,
        "grades": grades.to_dict(),
        "share_grade_1": float((qrels["relevance"] == 1).mean()),
        "tie_share": float((lane_rows["shape"] == "all_tied").mean()),
        "rows": len(lane_rows),
    })
audit = pd.DataFrame(records).sort_values("share_grade_1", ascending=False)
audit.round(3)

,lane,min_relevance,grades,share_grade_1,tie_share,rows
1,beir-nfcorpus,2,"{1: 11758, 2: 576}",0.953,0.061,165
4,crumb-set-operation-entity-retrieval,1,"{0: 35, 1: 3822, 2: 8828}",0.301,0.005,212
0,antique,3,"{1: 1642, 2: 2417, 3: 1196, 4: 1334}",0.249,0.000,159
5,dbpedia-entity,1,"{0: 28229, 1: 8785, 2: 6501}",0.202,0.061,279
3,crumb-paper-retrieval,1,"{0: 1467, 1: 1258, 2: 1259, 3: 1102, 4: 936, 5...",0.157,0.000,53
6,trec-dl-2022,2,"{0: 87318, 1: 15586, 2: 13760, 3: 542}",0.133,0.000,53
2,crumb-clinical-trial,1,"{0: 131516, 1: 22695, 2: 21552}",0.129,0.013,75


## 2 · Corpus-branch artifacts

Both builders are incremental per lane — a crashed run resumes where it stopped.
`GoldDocProfile` is the slow one: the full extractor over every unique gold document.

In [6]:
lanes = tuple(sorted(frame["dataset"].unique()))
corpus_profile = CorpusProfile(lanes).build()
corpus_profile.round(3)

,dataset,corpus.doc_count_log10,corpus.mean_words,corpus.p90_words,corpus.ttr,corpus.identifier_spans,corpus.nl_share
0,antique,4.511,41.598,85.0,0.114,0.940,0.406
1,beir-nfcorpus,3.560,220.979,299.8,0.055,18.379,0.286
2,bright-aops,4.000,148.762,310.1,0.049,27.700,0.256
3,bright-biology,4.000,59.023,97.0,0.138,3.423,0.200
4,bright-earth-science,4.000,81.869,85.0,0.182,10.201,0.079
5,bright-economics,4.000,66.250,196.0,0.122,5.938,0.161
6,bright-leetcode,4.000,105.484,228.1,0.098,17.226,0.149
7,bright-pony,3.897,41.988,80.0,0.061,6.936,0.081
8,bright-psychology,4.000,57.343,133.1,0.146,4.048,0.166
9,bright-robotics,4.000,29.947,79.0,0.172,7.834,0.118


In [7]:
gold_profile = GoldDocProfile(frame).build()
print(f"{len(gold_profile):,} query rows, "
      f"{gold_profile['gold.overlap'].isna().mean():.1%} without overlap")
gold_profile.head()

52,318 query rows, 1.4% without overlap


,dataset,query_id,gold.doc_count,gold.identifier_spans,gold.nl_share,gold.words,gold.overlap
0,antique,3990512,11.0,0.363636,0.373204,25.818182,0.5
1,antique,714612,11.0,0.181818,0.425900,25.909091,1.0
2,antique,2528767,16.0,4.375000,0.361858,63.750000,1.0
3,antique,821387,20.0,1.550000,0.389773,58.450000,0.9
4,antique,4448097,9.0,0.333333,0.358636,44.888889,1.0


## 3 · Target sanity

Cell targets are every predicate evaluated on every row — check which archetypes have
support, and that the two assumed column names exist before anything trains on them.

In [8]:
assert "natural_language_signal.natural_language_share" in table.catalog_rows.columns, (
    "NL scalar column name differs — fix encoder_router.targets.NL_SHARE"
)

cells = table.cell_targets
support = cells.sum().sort_values()
print(f"{int((support == 0).sum())} of {len(support)} cells with zero positive rows")
print(f"rows matching no cell: {(cells.sum(axis=1) == 0).mean():.1%}")
display(support.head(10).rename("thinnest"))
display(support.tail(10).rename("fattest"))

corpus_targets = table.corpus_targets()
corpus_targets.isna().mean().sort_values(ascending=False).head(8).rename("NaN share")

extracting features for 714 unindexed queries
1 of 44 cells with zero positive rows
rows matching no cell: 17.2%


legal_citation_canonical            0.0
datetime_token_present              1.0
single_token_char_blob              1.0
travel_transport_code               2.0
bibliographic_catalog_identifier    2.0
bio_clinical_identifier             2.0
business_temporal_reference         2.0
symbol_pile_no_grammar              2.0
standards_compliance_lookup         6.0
boolean_operator_query              7.0
Name: thinnest, dtype: float64

deep_nesting_single_sentence     3135.0
negation_bearing_question        3867.0
verbose_grammatical_request      3954.0
comparative_multi_entity         4495.0
short_grammatical_question       4637.0
wide_flat_enumeration            5416.0
extreme_length_pasted_query      5968.0
high_morphological_variation     8687.0
multi_statement_context_dump    10277.0
stopword_saturated_midlength    11440.0
Name: fattest, dtype: float64

gold.overlap              0.014194
gold.identifier_spans     0.014175
gold.nl_share             0.014175
gold.words                0.014175
gold.doc_count            0.000516
corpus.doc_count_log10    0.000000
corpus.mean_words         0.000000
corpus.p90_words          0.000000
Name: NaN share, dtype: float64

## 4 · Embedding caches

One forward pass per model, cached to parquet. The e5 cell feeds arm `e5_control` —
skip it if you are not running that arm yet.

In [9]:
embeddings = QueryEmbeddings(BGE).matrix(frame)
embeddings.shape

embedding 27 queries with BAAI/bge-small-en-v1.5


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.25it/s]


(52345, 384)

In [ ]:
QueryEmbeddings(E5).matrix(frame).shape

embedding 52318 queries with intfloat/multilingual-e5-small


Batches: 100%|██████████| 205/205 [00:56<00:00,  3.61it/s]


(52318, 384)

## 5 · One-fold smoke

The design arm against a single held lane — cheap proof that the whole path runs
before committing to the full sweep.

In [5]:
cv = LaneCV(table)
cv.run(Arm("design"), lanes=("beir-nfcorpus",))

design:   0%|          | 0/1 [00:00<?, ?it/s]

extracting features for 714 unindexed queries


design: 100%|██████████| 1/1 [00:35<00:00, 35.65s/it]


,arm,lane,rows,judged,threshold_sparse_only,threshold_dense_only,threshold_pure_rrf,served_dense_only,served_pure_rrf,served_sparse_only,serve_agreement,differ_agreement,captured,serve_captured,oracle,const_dense
0,design,beir-nfcorpus,138,116,0.35,0.4,0.5,0.310345,0.0,0.689655,0.689655,0.660377,0.52277,0.627067,0.639548,0.498679


## 6 · Full arm sweep

Every arm through every leave-one-lane-out fold. Results persist so readouts can be
re-run without re-training.

In [12]:
# A diverse 10-lane holdout panel: big/small, web/code/math/bio/legal-ish,
# identifier-dense and identifier-free. Training rows stay FULL per fold —
# only the number of readout points shrinks.
PANEL = (
    "beir-nfcorpus", "msmarco-passage-dev", "rarb-math",
    "crumb-code-retrieval", "gooaq", "scirgen-geo-en",
    "lotte-technology-forum", "quest", "freshstack-godot", "limit",
)
# _v3: captured-score threshold tuning (per-head), route pos_weight, and
# all_tied rows down-weighted to 0.25 in the route loss. _v2 = agreement-tuned
# shared threshold; unsuffixed = the collapsed 0.5 run. Both kept for reference.
RESULTS_PATH = OUT_DIR / "arm_results_panel__hedge_v3.parquet"

results = run_arms(table, lanes=PANEL, out_path=RESULTS_PATH)
results.head()

lightgbm: 100%|██████████| 10/10 [07:51<00:00, 47.15s/it]


embedding 27 queries with intfloat/multilingual-e5-small


feature_branch: 100%|██████████| 10/10 [03:42<00:00, 22.29s/it]


,arm,lane,rows,judged,threshold_sparse_only,threshold_dense_only,served_dense_only,served_pure_rrf,served_sparse_only,serve_agreement,differ_agreement,captured,serve_captured,oracle,const_dense,const_sparse,const_rrf
0,design,beir-nfcorpus,165.0,108.0,0.60,0.40,0.935185,0.0,0.064815,0.250000,0.255102,0.495354,0.612553,0.625959,0.496114,0.460112,0.591906
1,design,msmarco-passage-dev,3816.0,3717.0,0.55,0.35,0.969868,0.0,0.030132,0.299435,0.565925,0.813577,0.880447,0.889515,0.817931,0.622983,0.772542
2,design,rarb-math,2064.0,1820.0,0.55,0.30,0.932967,0.0,0.067033,0.251099,0.372474,0.664314,0.836396,0.848030,0.655813,0.648622,0.705281
3,design,crumb-code-retrieval,986.0,271.0,0.50,0.30,0.328413,0.0,0.671587,0.579336,0.578947,0.235136,0.457281,0.469386,0.375430,0.140919,0.285491
4,design,gooaq,7075.0,6390.0,0.55,0.35,0.967136,0.0,0.032864,0.266823,0.492148,0.761919,0.794172,0.812740,0.766205,0.567176,0.705175


## 7 · Readout

`differ_agreement` is the headline: agreement with the serve oracle exactly where the
routes disagree. The spread across lanes matters as much as the mean — ~15 lanes is
the real sample size. Read the design arm against `no_branches` (do the branches
help?), `shuffled_targets` (information or regularization?), and `lightgbm` (is the
MLP needed at all?).

In [13]:
results = pd.read_parquet(RESULTS_PATH)
wanted = [
    "differ_agreement", "serve_agreement", "captured", "serve_captured",
    "const_dense", "oracle",
] + [c for c in results.columns if c.startswith(("threshold_", "served_"))]
present = [c for c in wanted if c in results.columns]
summary = results.groupby("arm")[present].mean()
summary.insert(1, "differ_spread", results.groupby("arm")["differ_agreement"].std())
summary.sort_values("captured", ascending=False).round(4)

,differ_agreement,differ_spread,serve_agreement,captured,serve_captured,const_dense,oracle,threshold_sparse_only,threshold_dense_only,served_dense_only,served_pure_rrf,served_sparse_only
arm,,,,,,,,,,,,
lightgbm,0.3300,0.1769,0.2570,0.4829,0.6832,0.4815,0.6968,0.520,0.735,0.9040,0.0000,0.0960
zipf_channel,0.3684,0.1656,0.3081,0.4804,0.6832,0.4815,0.6968,0.545,0.315,0.8304,0.0000,0.1696
e5_control,0.3860,0.2046,0.3215,0.4752,0.6832,0.4815,0.6968,0.545,0.310,0.7894,0.0000,0.2106
shuffled_targets,0.3493,0.1662,0.2817,0.4749,0.6832,0.4815,0.6968,0.520,0.335,0.8680,0.0000,0.1320
no_branches,0.3745,0.1674,0.3137,0.4735,0.6832,0.4815,0.6968,0.550,0.355,0.8161,0.0000,0.1839
feature_branch,0.3590,0.1634,0.2979,0.4667,0.6832,0.4815,0.6968,0.565,0.345,0.8245,0.0000,0.1755
design,0.3242,0.1816,0.2572,0.4640,0.6832,0.4815,0.6968,0.570,0.340,0.8699,0.0000,0.1301
features_as_input,0.3368,0.1839,0.2725,0.4584,0.6832,0.4815,0.6968,0.605,0.335,0.8270,0.0001,0.1729


import matplotlib.pyplot as plt

arms = sorted(results["arm"].unique())
fig, ax = plt.subplots(figsize=(9, 4))
for i, arm in enumerate(arms):
    rows = results[results["arm"] == arm]
    ax.scatter([i] * len(rows), rows["differ_agreement"], alpha=0.5)
    ax.scatter([i], [rows["differ_agreement"].mean()], color="black", marker="_", s=400)
ax.set_xticks(range(len(arms)), arms, rotation=20)
ax.set_ylabel("differ_agreement per held lane")
ax.set_title("leave-one-lane-out spread (mean marked)")
plt.tight_layout()
plt.show()

In [9]:
# paired per-lane deltas on CAPTURED score — the metric the tuner now
# optimizes; agreement rewards the majority-class constant by construction.
wide = results.pivot(index="lane", columns="arm", values="captured")
wide["constant_sparse"] = (
    frame[frame["serve"].notna()]
    .groupby("dataset")["score_sparse_only"].mean()
)
pairs = {
    "no_branches - design": wide["no_branches"] - wide["design"],
    "design - shuffled": wide["design"] - wide["shuffled_targets"],
    "e5 - design": wide["e5_control"] - wide["design"],
    "best_arm - constant": (
        wide.drop(columns="constant_sparse").max(axis=1)
        - wide["constant_sparse"]
    ),
}
pd.DataFrame(
    {k: {"mean": v.mean(), "lanes_won": (v > 0).sum()} for k, v in pairs.items()}
).T

,mean,lanes_won
no_branches - design,0.008151,6.0
design - shuffled,0.002970,6.0
e5 - design,-0.046300,3.0
best_arm - constant,0.005123,8.0


## 8 · Archetype probes

The seven standing probes through a design-arm router trained on all rows (no holdout —
this is the smoke test, not an evaluation).

In [ ]:
from sentence_transformers import SentenceTransformer

from encoder_router.evaluate import TIE_WEIGHT, tuned_thresholds
from encoder_router.model import EncoderRouter
from encoder_router.table import EMBEDDING_PREFIXES
from hybrid_search_rrf_dataset.probes import ARCHETYPE_PROBES

everything = np.ones(len(frame), dtype=bool)
svd = NgramSvd().fit(frame["query"])
x_all = np.concatenate([embeddings, svd.transform(frame["query"])], axis=1)
route = table.route_targets().to_numpy(dtype=np.float32)
cell = table.cell_targets.to_numpy(dtype=np.float32)
joined = pd.concat([table.corpus_targets(), table.outcome_rates(everything)], axis=1)
corpus = ((joined - joined.mean()) / joined.std().replace(0.0, 1.0).fillna(1.0)).to_numpy(np.float32)
TIE_POLICY = 1.0
tie_weights = np.where(frame["shape"] == "all_tied", TIE_POLICY, 1.0).astype(np.float32)

router = EncoderRouter().fit(x_all, route, cell, corpus, route_weights=tie_weights)
thresholds = tuned_thresholds(router.probabilities(x_all), frame)
print("tuned thresholds (sparse, dense) — rrf is the hedge when neither fires:",
      thresholds.round(2))

queries = [p.query for p in ARCHETYPE_PROBES]
encoder = SentenceTransformer(BGE)
probe_emb = encoder.encode(
    [EMBEDDING_PREFIXES[BGE] + q for q in queries], normalize_embeddings=True
)
probe_x = np.concatenate([probe_emb, svd.transform(pd.Series(queries))], axis=1)
served = router.predict_routes(probe_x.astype(np.float32), threshold=thresholds)

pd.DataFrame({
    "query": queries,
    "expected": [p.expected for p in ARCHETYPE_PROBES],
    "served": served,
    "agrees": [
        None if p.expected is None else str(p.expected) == s
        for p, s in zip(ARCHETYPE_PROBES, served)
    ],
})

In [17]:
probs = router.probabilities(probe_x.astype(np.float32)).round(3)
probs.insert(0, "query", [q[:40] for q in queries])
probs

,query,dense_only,pure_rrf,sparse_only
0,a3f5d8b9e12c4d56789abcdef0123456,0.666,0.513,0.399
1,/etc/nginx/nginx.conf,0.645,0.550,0.461
2,ERR_CONNECTION_RESET,0.708,0.557,0.417
3,explain quicksort,0.587,0.507,0.447
4,HTTP 502,0.664,0.552,0.446
5,comment volent les oiseaux,0.609,0.538,0.473
6,como aprender a programar en rust,0.601,0.536,0.472


### 8b · Is the tuned `t_dense` a peak or a plateau?

Sweep `t_dense` while holding the tuned `t_sparse`, scoring each value with the
tuner's own cost-adjusted reward on the training differ rows. Reading the
`gap_to_best` column:

- **flat plateau** (several values within ~0.002 of the best): the tuner's pick
  is one point on a ridge — choosing the plateau's *low* edge is statistically
  free, and it lets mid-confidence dense beliefs (the 0.50-ish probes) serve.
- **steep peak**: the training data genuinely objects to a lower bar; the
  probes wait for v3 labels to re-price mid-confidence dense.

Honesty rail: never pick a threshold *because the probes look better* — that is
fitting the instrument. Only a flat plateau licenses the choice, and then only
as a tie-break within noise.

In [14]:
from encoder_router.evaluate import COST_STEP
from encoder_router.table import HEAD_ROUTES, HEDGE, PRIORITY, serve_indices

differ = ((frame["shape"] == "routes_differ") & frame["serve"].notna()).to_numpy()
choices = [*HEAD_ROUTES, HEDGE]
rewards = (
    frame[[f"score_{r}" for r in choices]].to_numpy()[differ]
    - COST_STEP * np.array([PRIORITY.index(r) for r in choices])
)
ordered = router.probabilities(x_all)[list(HEAD_ROUTES)].to_numpy()[differ]

rows_ = []
for td in np.arange(0.30, 0.71, 0.05):
    trial = np.array([float(thresholds[0]), round(float(td), 2)])
    pick = serve_indices(ordered, trial)
    rows_.append({
        "t_dense": round(float(td), 2),
        "reward": float(rewards[np.arange(len(pick)), pick].mean()),
        "dense_share": float(
            (pick == list(HEAD_ROUTES).index("dense_only")).mean()
        ),
    })
curve = pd.DataFrame(rows_)
curve["gap_to_best"] = (curve["reward"].max() - curve["reward"]).round(4)
curve.round(4)

NameError: name 'router' is not defined

### 8c · Checkpoint the fitted router

A serving checkpoint is four artifacts: the trained net (`router.pt`), the
fitted n-gram SVD basis (`ngram_svd.joblib` — inputs are meaningless without
the exact basis), the tuned thresholds, and a `meta.json` recording when and
on what. Run this whenever a §8 fit is worth keeping — training is seeded and
reproducible, but a checkpoint survives kernel deaths and code changes alike.
The default path is overwritten on rerun; rename `CKPT` to keep several.

In [ ]:
import json
from datetime import datetime, timezone

CKPT = OUT_DIR / "checkpoints" / "probe_router"

router.save(CKPT / "router.pt")
svd.save(CKPT / "ngram_svd.joblib")
np.save(CKPT / "thresholds.npy", thresholds)
(CKPT / "meta.json").write_text(json.dumps({
    "saved_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "embedding_model": BGE,
    "tie_policy": float(TIE_POLICY),
    "rows_trained": int(len(frame)),
    "thresholds": [float(t) for t in thresholds],
}, indent=2))
print("checkpoint written:")
for artifact in sorted(CKPT.iterdir()):
    print(f"  {artifact.name}")

# load-back, any later session (plus QueryEmbeddings(BGE) for the embedding):
#   from encoder_router.model import EncoderRouter
#   from encoder_router.table import NgramSvd
#   router = EncoderRouter.load(CKPT / "router.pt")
#   svd = NgramSvd.load(CKPT / "ngram_svd.joblib")
#   thresholds = np.load(CKPT / "thresholds.npy")

## 9 · Feature recoverability probe (diagnostic, zero gradient)

A ridge regression from the model's inputs to each taxonomy feature. Near-zero R²
means the feature is absent from the inputs — the only kind of feature that could
earn input status, and evidence for a rarity channel if the sparse-signals cluster
at the bottom. Nothing here touches the model.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

features = table.feature_matrix
x_tr, x_te, f_tr, f_te = train_test_split(x_all, features, test_size=0.25, random_state=0)
rows = []
for column in features.columns:
    if f_tr[column].std() == 0:
        continue
    score = Ridge(alpha=1.0).fit(x_tr, f_tr[column]).score(x_te, f_te[column])
    rows.append({"feature": column, "r2": score})
recoverability = pd.DataFrame(rows).sort_values("r2").reset_index(drop=True)
print("least recoverable (candidates for input status):")
display(recoverability.head(12))
print("most recoverable (provably present in the inputs):")
recoverability.tail(12)

## What to bring back

- design vs `no_branches` on `differ_agreement` — do the branches earn their place?
- design vs `shuffled_targets` — if they tie, the gain is regularization, not information.
- `lightgbm` vs design — if the trees match, ship the trees.
- `e5_control` vs design on holdout — the label-coupling verdict.

Not in this notebook: the fine-tuned-bge ceiling arm (own training loop, one run,
add here when wanted) and the cross-lane near-dup guard (open in TODOS). All of it
re-runs against v3 for the numbers of record.